# "See & Remember" — Hackathon Starter Notebook

**Deep Learning Chapters 9 (Convolutional Networks) & 10 (Recurrent Networks)**

This single notebook runs both flagship challenges end-to-end and stubs out the bonus challenges.
Everything is small enough to train in seconds to ~2 minutes.

### Before you start (30 seconds)
1. **Turn on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator → GPU → Save**.
2. Run the **Setup** cell once.
3. Jump to your flagship: **Flagship A (CNN)** or **Flagship B (RNN)**.

### How to read this notebook
- Cells run top-to-bottom and work as-is, so you always have something that runs.
- Look for **`# >>> EXPERIMENT`** blocks — that is where you change the architecture, add a
  challenge, or push for a better result. The base points are for *understanding and explaining*
  what happens, not just for a number.

### Reference
- CNN chapter: https://ketanp23.github.io/deeplearningclass/cnn-deep-learning.html
- RNN chapter: https://ketanp23.github.io/deeplearningclass/rnn-deep-learning.html


## Setup — run this once

In [ ]:
import time, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)
if device == "cpu":
    print("No GPU detected. It will still run, just slower. "
          "Runtime -> Change runtime type -> GPU to speed up.")

---
# Flagship A — CNN "Dense vs Conv Showdown"

**Claim to prove (chapter stations 1-4, 6):** a convolutional net matches or beats a dense net on
images while using *far fewer parameters* and *losing less accuracy when the image shifts* — because
of sparse connections, shared weights, and pooling.

**Deliverables**
1. CNN test accuracy **>= 90%** on Fashion-MNIST.
2. Parameter count: CNN vs dense net.
3. **Shift-robustness:** accuracy of each model when every test image is rolled by a few pixels.


### A1 — Load Fashion-MNIST

In [ ]:
from torchvision import datasets, transforms

tfm = transforms.ToTensor()  # values in [0,1], shape (1,28,28)
train_ds = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

classes = ["T-shirt","Trouser","Pullover","Dress","Coat",
           "Sandal","Shirt","Sneaker","Bag","Ankle boot"]
print("train:", len(train_ds), "| test:", len(test_ds))

### A2 — Two models: a dense net and a small CNN

In [ ]:
class DenseNet(nn.Module):
    """Fully-connected baseline. Every pixel wired to every hidden unit."""
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, hidden), nn.ReLU(),
            nn.Linear(hidden, 10),
        )
    def forward(self, x):
        return self.net(x)


class SmallCNN(nn.Module):
    """Local + shared filters, then pooling. Toggle use_bn / use_pool for the bonuses."""
    def __init__(self, use_bn=False, use_pool=True):
        super().__init__()
        def block(cin, cout):
            layers = [nn.Conv2d(cin, cout, kernel_size=3, padding=1)]
            if use_bn:   layers.append(nn.BatchNorm2d(cout))
            layers.append(nn.ReLU())
            if use_pool: layers.append(nn.MaxPool2d(2))
            return layers
        self.features = nn.Sequential(*block(1,16), *block(16,32))
        with torch.no_grad():
            flat = self.features(torch.zeros(1,1,28,28)).flatten(1).shape[1]
        self.head = nn.Linear(flat, 10)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### A3 — Train / evaluate helpers

In [ ]:
def train(model, loader, epochs=2, lr=1e-3):
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    t0 = time.time()
    for ep in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward()
            opt.step()
        print(f"  epoch {ep+1}/{epochs}  loss {loss.item():.3f}")
    print(f"  trained in {time.time()-t0:.1f}s")
    return model

@torch.no_grad()
def accuracy(model, loader, pixel_shift=0):
    model.eval()
    correct = total = 0
    for xb, yb in loader:
        if pixel_shift:                    # roll the image to test translation robustness
            xb = torch.roll(xb, shifts=(pixel_shift, pixel_shift), dims=(2,3))
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total   += yb.size(0)
    return correct / total

### A4 — Run the showdown

In [ ]:
print("Dense net"); dense = train(DenseNet(), train_loader, epochs=2)
print("CNN");       cnn   = train(SmallCNN(), train_loader, epochs=2)

for name, m in [("Dense", dense), ("CNN", cnn)]:
    acc  = accuracy(m, test_loader)
    accs = accuracy(m, test_loader, pixel_shift=4)   # 4-pixel diagonal shift
    print(f"\n{name:5s} | params {count_params(m):>8,} | "
          f"acc {acc:.3f} | acc@shift4 {accs:.3f} | drop {acc-accs:+.3f}")

**Read the result.** Typically the CNN has ~10x fewer parameters *and* a smaller accuracy drop
under shift. That drop is the whole point of stations 3-4: pooling + weight sharing buy translation
tolerance the dense net has to learn from scratch.

```
# >>> EXPERIMENT
#  - Can you get the CNN past 92% (more epochs? a third conv block? padding tweaks?)
#  - Make the dense net WIDER until it matches CNN accuracy - now compare parameter counts.
#  - Try pixel_shift = 2, 6, 8. Where does each model break?
```

### Bonus A — Gabor hunt (+15) & pooling ablation (+15) & batch-norm (+10)

**Gabor hunt:** the chapter says first-layer filters learn to look like oriented edge / Gabor
detectors. Plot yours and judge.

In [ ]:
w = cnn.features[0].weight.data.cpu()   # (16,1,3,3): first conv's 16 filters
fig, axes = plt.subplots(2, 8, figsize=(11,3))
for i, ax in enumerate(axes.flat):
    ax.imshow(w[i,0], cmap="gray"); ax.axis("off")
fig.suptitle("First-layer learned filters — which look like oriented edge detectors?")
plt.show()

**Pooling ablation:** remove pooling and see how much shift-robustness you lose.

In [ ]:
# >>> EXPERIMENT: train a no-pooling CNN and compare its accuracy drop under shift.
cnn_nopool = train(SmallCNN(use_pool=False), train_loader, epochs=2)
a  = accuracy(cnn_nopool, test_loader)
a4 = accuracy(cnn_nopool, test_loader, pixel_shift=4)
print(f"no-pool CNN | params {count_params(cnn_nopool):,} | "
      f"acc {a:.3f} | acc@shift4 {a4:.3f} | drop {a-a4:+.3f}")
print("Compare this drop with the pooling CNN above.")

**BatchNorm on/off:** compare how fast the loss falls with `use_bn=True` vs `False` (station 10).

In [ ]:
# >>> EXPERIMENT: train SmallCNN(use_bn=True) vs SmallCNN(use_bn=False),
# log the per-step loss for each, and plot both curves on one axis.
# Hint: copy train() and append loss.item() to a list each step instead of only printing.

**Separable-conv speedup (+10, pure timing, no training):** a k*k blur done as two 1-D passes
costs 2k multiplies per pixel instead of k*k. Measure it.

In [ ]:
# >>> EXPERIMENT
# k = 9
# naive: one Conv2d(1,1,k,padding=k//2) ; separable: Conv2d(1,1,(1,k)) then Conv2d(1,1,(k,1))
# Time each on a big random image with torch.no_grad(); compare measured speedup to the predicted k/2.

---
# Flagship B — RNN "Memory Showdown"

**Claim to prove (chapter stations 2, 3, 6, 7, 8):** a plain RNN forgets across long gaps
(vanishing gradient), while a gated cell (LSTM/GRU) carries information across them thanks to its
additive memory path.

**The task (the chapter's recall task).** Show a signal token at the *first* step, then a run of
blanks. At the *last* step the network must reproduce the first token. The gap = sequence length - 1.
Chance accuracy is 1 / (number of signal tokens).

**Deliverable:** accuracy vs gap for a plain RNN and an LSTM on the same task; show where the RNN
collapses toward chance.


### B1 — The recall task

In [ ]:
N_SIGNALS = 8              # signal tokens 0..7 ; token 8 is the "blank"
BLANK = N_SIGNALS
VOCAB = N_SIGNALS + 1

def make_batch(batch_size, seq_len):
    x = torch.full((batch_size, seq_len), BLANK, dtype=torch.long)
    signal = torch.randint(0, N_SIGNALS, (batch_size,))
    x[:, 0] = signal                       # the thing to remember sits at step 0
    return x.to(device), signal.to(device) # label = that first token

xb, yb = make_batch(4, 6)
print("example inputs (8 = blank):\n", xb.cpu())
print("labels (recall these):", yb.cpu().tolist(),
      "| chance accuracy =", round(1/N_SIGNALS, 3))

### B2 — One model, swappable cell

In [ ]:
class Recaller(nn.Module):
    def __init__(self, cell="rnn", hidden=64):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        if cell == "rnn":
            self.rnn = nn.RNN(32, hidden, batch_first=True, nonlinearity="tanh")
        elif cell == "lstm":
            self.rnn = nn.LSTM(32, hidden, batch_first=True)
        elif cell == "gru":
            self.rnn = nn.GRU(32, hidden, batch_first=True)
        self.head = nn.Linear(hidden, N_SIGNALS)

    def forward(self, x):
        out, _ = self.rnn(self.embed(x))   # out: (batch, seq, hidden)
        return self.head(out[:, -1, :])    # read the state at the LAST step

### B3 — Train across gaps and plot

In [ ]:
def train_recaller(cell, seq_len, iters=400, lr=2e-3, clip=None):
    model = Recaller(cell).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    for _ in range(iters):
        x, y = make_batch(64, seq_len)
        opt.zero_grad()
        loss = lossf(model(x), y)
        loss.backward()
        if clip is not None:                       # gradient clipping (bonus / station 9)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
    x, y = make_batch(2000, seq_len)               # eval on fresh data
    with torch.no_grad():
        acc = (model(x).argmax(1) == y).float().mean().item()
    return acc

gaps = [1, 10, 25, 50]        # gap = seq_len - 1
rnn_acc, lstm_acc = [], []
for L in gaps:
    r = train_recaller("rnn",  L+1)
    l = train_recaller("lstm", L+1)
    rnn_acc.append(r); lstm_acc.append(l)
    print(f"gap {L:3d} | plain RNN {r:.3f} | LSTM {l:.3f}")

plt.figure(figsize=(6,4))
plt.plot(gaps, rnn_acc,  "o-", label="plain RNN")
plt.plot(gaps, lstm_acc, "s-", label="LSTM")
plt.axhline(1/N_SIGNALS, ls="--", c="gray", label="chance")
plt.xlabel("gap (steps between signal and recall)"); plt.ylabel("test accuracy")
plt.title("Memory Showdown: who remembers across the gap?"); plt.legend(); plt.show()

**Read the result.** The LSTM stays near-perfect as the gap grows; the plain RNN slides toward
chance. That collapse is the vanishing gradient of station 6, and the LSTM's flat line is the
constant error carousel of station 7.

```
# >>> EXPERIMENT
#  - Add "gru" to the race. Does it track the LSTM?
#  - Push gaps to [1, 25, 75, 150]. Does the RNN ever recover with more iters?
#  - Re-run the plain RNN with clip=1.0 - does clipping fix vanishing? (It shouldn't: clipping
#    treats EXPLODING gradients. Explain why in your write-up.)
```

### Bonus B — gradient-norm evidence (+ points), char generator (+15), clip-the-cliff (+10)

**Gradient-norm evidence:** measure how much gradient from the final loss reaches step 0 as the gap
grows — direct evidence of vanishing.

In [ ]:
# >>> EXPERIMENT
# For an UNTRAINED Recaller("rnn"), run one forward+backward on a batch at each seq_len,
# then read model.embed.weight.grad norm (gradient that reached the input at step 0).
# Plot grad-norm vs gap: it should shrink fast for "rnn" and stay healthier for "lstm".

**Char-level generator (one-to-many, teacher forcing):** train a tiny char RNN and sample from it.

In [ ]:
# >>> EXPERIMENT (optional, fun)
# text = "to be or not to be that is the question " * 200
# Build a char vocab, train an LSTM to predict the next char (teacher forcing = feed true chars),
# then generate by feeding the model its own output. Note where teacher forcing happens.

**Clip-the-cliff (+10):** reproduce an exploding gradient, then tame it with clipping.

In [ ]:
# >>> EXPERIMENT
# Train a plain RNN on a long sequence with a large lr (e.g. 0.1) and log the gradient norm each step.
# You should see occasional huge spikes. Re-run with nn.utils.clip_grad_norm_(..., 1.0) and compare
# the loss curves. This is station 9's gradient clipping.

---
# Concept track — boss battle (no training required)

Answer in a text cell or on paper. Each is worth points for a correct, *explained* answer.

1. Why does a fully-connected layer scale badly on images, and which three CNN properties fix it?
2. What does a single value in a feature map measure? What is a kernel?
3. Name the three stages of a convolutional layer. Why does pooling give approximate translation
   invariance?
4. Relate V1 simple cells and complex cells to convolution and pooling. What is a Gabor filter?
5. "Convolution is a dense layer with an infinitely strong prior." State the two constraints of that
   prior, and what pooling's prior asserts.
6. Define stride, padding, and dilation. Give the output-size formula.
7. What is a structured output? How does convolution's spatial preservation enable segmentation?
8. Write the vanilla RNN cell equations. What does the hidden state represent, and what is parameter
   sharing across time?
9. In BPTT, why is a shared weight's gradient a sum over time steps, and where does the product of
   Jacobians appear?
10. List the five RNN input-output types with one task each.
11. Why can't a bidirectional RNN be used for streaming / causal prediction?
12. Describe the LSTM's forget, input, and output gates and the additive cell update. Why is the
    additive path called the constant error carousel? How does the GRU differ?


---
## Submission checklist
- [ ] Flagship result cell runs top-to-bottom and prints/plots your numbers.
- [ ] One or two sentences of **why**, in the chapter's language, in a text cell.
- [ ] Any bonus cells you attempted, clearly labelled.
- [ ] A 90-second demo ready: one result + one "why".

*Good luck. Slide the filter, unroll the loop.*